# Classificação de Áudio a partir de Embeddings

Este notebook classifica gravações de áudio aplicando um **modelo classificador** sobre embeddings pré-computados armazenados em um banco de dados SQLite.

Este é o segundo passo de um pipeline em dois estágios:
1. **Banco de Dados de Embeddings** — extrai embeddings de um backbone de modelo de fundação
2. **Este notebook** — aplica um classificador leve sobre esses embeddings

---

### O que este notebook faz (em ordem):
1. **Conecta ao seu Google Drive** para ler o banco de embeddings e salvar resultados
2. **Instala o software necessário** automaticamente
3. **Carrega** o modelo classificador, o arquivo de rótulos e os metadados do banco
4. **Verifica** quais gravações já foram classificadas
5. **Executa a classificação** em cada segmento de áudio e salva os resultados no Google Drive
6. **Mescla** todos os resultados por gravação em um único arquivo CSV

### Modelo classificador:
O modelo deve aceitar um vetor de embedding como entrada e retornar logits brutos (um valor por classe).
Formatos suportados: **ONNX** (`.onnx`) e **TFLite** (`.tflite`).
O modelo pode ser carregado do Google Drive ou baixado do HuggingFace.

### Arquivos de resultado:
Um arquivo `.results.txt` é salvo por gravação, em `PASTA_RESULTADOS / NOME_PONTO /`. Manter um arquivo
por gravação permite que uma execução interrompida ou desconectada do Colab retome de onde parou (o
Passo 4 ignora gravações que já têm arquivo de resultado). O Passo 6 então mescla todos eles em um
**único CSV** (`PASTA_RESULTADOS / MODEL_NAME.merged.results.csv`), pronto para abrir diretamente no
notebook *Análise de Resultados de Classificação*.
Colunas: `stream_id`, `site`, `lat`, `lon`, `date`, `time`, `utc_offset`, `start_time`, `end_time`, `label`, `confidence`.

### Como executar:
Execute cada célula **uma de cada vez**, de cima para baixo. Clique em ▶ ou pressione `Shift + Enter`.

> **Novo em notebooks?** Uma célula com `[ ]` ainda não foi executada. Após executar aparece `[1]`, `[2]`, etc.

---

Criado pela [biodiversica](https://biodiversica.xyz). Para problemas, dúvidas ou feedback, abra uma issue no [GitHub](https://github.com/biodiversica/bioacoustic-ipynbs/issues) ou visite [biodiversica.xyz](https://biodiversica.xyz).

---
## Passo 1 — Conectar ao Google Drive e Instalar Software

Execute as duas células abaixo. A primeira pedirá que você **permita acesso ao seu Google Drive**.

In [ ]:
#@title 📂 Conectar ao Google Drive { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive conectado com sucesso.')

In [ ]:
#@title 📦 Instalar pacotes { display-mode: "form" }

!pip install ai-edge-litert onnxruntime huggingface_hub -q

print('\nTodos os pacotes instalados com sucesso.')

---
## Passo 2 — Configuração

Preencha os quatro formulários abaixo e execute-os de cima para baixo.

1. **Configurações Gerais** — pasta de resultados, limiar de confiança, top-K, coordenadas dos pontos
2. **Banco de Embeddings** — caminho para o arquivo `.db` e prefixo de nome de arquivo opcional
3. **Modelo Classificador** — arquivo do modelo e configurações de inferência
4. **Rótulos** — arquivo de rótulos e configurações de leitura

> **Dica:** Os formulários ocultam o código automaticamente. Clique em `{ }` no canto superior direito de qualquer célula para ver o código por baixo.

In [ ]:
#@title ⚙️ Configurações Gerais { display-mode: "form" }

import os

#@markdown **Pasta de resultados** — onde os resultados de detecção serão salvos no seu Google Drive.
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/classification_results'  #@param {type:"string"}

#@markdown ---
#@markdown **Limiar de detecção** — pontuação mínima de confiança para manter uma detecção (0.0–1.0).
#@markdown Menor = mais detecções (pode incluir falsos positivos). Maior = menos, porém mais confiáveis.
MIN_CONFIDENCE = 0.25  #@param {type:"slider", min:0.0, max:1.0, step:0.05}

#@markdown ---
#@markdown **Top K detecções por segmento** — máximo de detecções a manter por segmento de áudio.
#@markdown 1 = apenas a detecção de maior confiança. Aumente para manter mais.
TOP_K = 1  #@param {type:"integer"}

#@markdown ---
#@markdown **Coordenadas por ponto** (opcional) — latitude e longitude por ponto de gravação.
#@markdown Formato: `NOME_PONTO=lat,lon` separados por ponto e vírgula.
#@markdown Exemplo: `PONTO_A=-15.1,-47.2;PONTO_B=-16.3,-48.1`
SITE_COORDINATES = ''  #@param {type:"string"}

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

_latlon_map = {}
for _e in SITE_COORDINATES.split(';'):
    _e = _e.strip()
    if '=' in _e:
        _n, _c = _e.split('=', 1)
        _p = _c.split(',')
        try:
            _latlon_map[_n.strip()] = (float(_p[0]), float(_p[1]))
        except Exception:
            pass

print(f'Pasta de resultados : {DRIVE_RESULTS_DIR}')
print(f'Confiança mínima    : {MIN_CONFIDENCE}')
print(f'Top K               : {TOP_K}')
if _latlon_map:
    for _sn, (_la, _lo) in _latlon_map.items():
        print(f'  Coordenadas     : {_sn} → lat={_la}, lon={_lo}')

In [ ]:
#@title 🗄️ Banco de Embeddings { display-mode: "form" }

import os

#@markdown **Caminho do banco de dados** — caminho completo para o arquivo `.db` criado pelo notebook *Banco de Dados de Embeddings*.
EMBEDDINGS_DB_PATH = '/content/drive/MyDrive/embeddings/my_model.embeddings.db'  #@param {type:"string"}

#@markdown ---
#@markdown **Prefixo do nome de arquivo** — usado para extrair datas/horas das chaves de embedding.
#@markdown Deixe em branco para nomes de arquivo AudioMoth padrão (`YYYYMMDD_HHMMSS.wav`).
#@markdown Digite `SM4` se os arquivos foram nomeados como `SM4_20250615_203000.wav`.
FILENAME_PREFIX = ''  #@param {type:"string"}

FILENAME_PREFIX = FILENAME_PREFIX.strip()

if not os.path.exists(EMBEDDINGS_DB_PATH):
    print(f'AVISO: Banco de dados não encontrado: {EMBEDDINGS_DB_PATH}')
    print('Verifique o caminho acima — certifique-se de que o Google Drive está conectado.')
else:
    import sqlite3
    with sqlite3.connect(EMBEDDINGS_DB_PATH) as _c:
        _n = _c.execute('SELECT COUNT(*) FROM embeddings').fetchone()[0]
    print(f'Banco de dados     : {EMBEDDINGS_DB_PATH}')
    print(f'Embeddings salvos  : {_n}')

In [ ]:
#@title 🤖 Modelo Classificador { display-mode: "form" }

#@markdown **Fonte do modelo** — onde o arquivo do modelo classificador está armazenado.
MODEL_SOURCE = 'google_drive'  #@param ["google_drive", "huggingface"]

#@markdown ---
#@markdown ### Se fonte = Google Drive
#@markdown Caminho completo para o arquivo do modelo no Drive (`.onnx` ou `.tflite`).
DRIVE_MODEL_PATH = '/content/drive/MyDrive/models/classifier_head.onnx'  #@param {type:"string"}

#@markdown ---
#@markdown ### Se fonte = HuggingFace
HF_REPO_ID    = ''  #@param {type:"string"}
HF_MODEL_FILE = 'classifier_head.onnx'  #@param {type:"string"}

#@markdown ---
#@markdown **Função de ativação** — como o modelo converte saídas brutas em pontuações de confiança.
#@markdown `sigmoid` = cada classe de forma independente (mais comum). `softmax` = pontuações somam 1. `none` = modelo já retorna probabilidades.
ACTIVATION = 'sigmoid'  #@param ["sigmoid", "softmax", "none"]

#@markdown ---
#@markdown **Sensibilidade do sigmoid** — inclinação da curva sigmoid (usado apenas quando ativação = `sigmoid`). `-1.0` = sigmoid padrão; mais negativo = mais íngreme.
SIGMOID_SENSITIVITY = -1.0  #@param {type:"number"}
#@markdown **Viés do sigmoid** — deslocamento horizontal do sigmoid (usado apenas quando ativação = `sigmoid`). `1.0` = sem deslocamento; acima de 1.0 = mais sensível (pontuações maiores); abaixo de 1.0 = mais conservador (pontuações menores).
SIGMOID_BIAS = 1.0  #@param {type:"slider", min:0.01, max:1.99, step:0.01}

#@markdown ---
#@markdown ### Configurações ONNX *(ignoradas para modelos TFLite)*
#@markdown **Nome do nó de entrada** — nome do tensor de entrada ONNX (o vetor de embedding).
ONNX_INPUT_NAME = 'embedding'  #@param {type:"string"}
#@markdown **Nome do nó de saída** — nome do tensor de saída ONNX (os logits).
ONNX_OUTPUT_NAME = 'logits'  #@param {type:"string"}

print(f'Fonte do modelo  : {MODEL_SOURCE}')
print(f'Ativação         : {ACTIVATION}')
print(f'Sigmoid      : sensibilidade={SIGMOID_SENSITIVITY}  viés={SIGMOID_BIAS}  (usado apenas quando ativação = sigmoid)')
if MODEL_SOURCE == 'google_drive':
    print(f'Caminho do modelo: {DRIVE_MODEL_PATH}')
else:
    print(f'HF repo          : {HF_REPO_ID}')
    print(f'HF file          : {HF_MODEL_FILE}')
if ONNX_INPUT_NAME or ONNX_OUTPUT_NAME:
    print(f'ONNX entrada     : {ONNX_INPUT_NAME}')
    print(f'ONNX saída       : {ONNX_OUTPUT_NAME}')

In [ ]:
#@title 🏷️ Rótulos { display-mode: "form" }

#@markdown **Fonte dos rótulos** — onde o arquivo de rótulos está armazenado.
LABELS_SOURCE = 'google_drive'  #@param ["google_drive", "huggingface"]

#@markdown ---
#@markdown ### Se fonte = Google Drive
#@markdown Caminho completo para o arquivo de rótulos no Drive (texto simples, um rótulo por linha).
DRIVE_LABELS_PATH = '/content/drive/MyDrive/models/labels.txt'  #@param {type:"string"}

#@markdown ---
#@markdown ### Se fonte = HuggingFace
#@markdown Deixe **HF labels repo** em branco para usar o mesmo repositório do modelo.
HF_LABELS_REPO = ''  #@param {type:"string"}
HF_LABELS_FILE = 'labels.txt'  #@param {type:"string"}

#@markdown ---
#@markdown ### Configurações do arquivo de rótulos
#@markdown **Tem linha de cabeçalho?** — marque se a primeira linha é um cabeçalho de coluna, não um rótulo.
LABELS_HAS_HEADER = False  #@param {type:"boolean"}
#@markdown **Índice da coluna de rótulo** — qual coluna contém o rótulo (0 = primeira coluna).
LABELS_COLUMN_INDEX = 0  #@param {type:"integer"}
#@markdown **Separador de colunas** — como as colunas são separadas no arquivo de rótulos.
LABELS_DELIMITER = 'underscore (_)'  #@param ["tab", "comma (,)", "semicolon (;)", "underscore (_)"]

print(f'Fonte dos rótulos  : {LABELS_SOURCE}')
if LABELS_SOURCE == 'google_drive':
    print(f'Caminho dos rótulos: {DRIVE_LABELS_PATH}')
else:
    print(f'HF labels repo     : {HF_LABELS_REPO or "(mesmo repo do modelo)"}')
    print(f'HF labels file     : {HF_LABELS_FILE}')
print(f'Tem cabeçalho      : {LABELS_HAS_HEADER}')
print(f'Índice da coluna   : {LABELS_COLUMN_INDEX}')
print(f'Separador          : {LABELS_DELIMITER}')

---
## Passo 3 — Carregar Modelo, Rótulos e Metadados do Banco

Esta célula baixa o classificador (se vier do HuggingFace), carrega o modelo e o arquivo de rótulos,
e lê os parâmetros de embedding armazenados no banco de dados (tamanho da janela, tamanho do embedding, taxa de amostragem).

In [ ]:
#@title 🧠 Carregar modelo, rótulos e metadados do banco { display-mode: "form" }

import os
import sqlite3
import numpy as np

# --- Carrega modelo classificador ---
if MODEL_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    print(f'Baixando modelo do HuggingFace: {HF_REPO_ID} / {HF_MODEL_FILE} ...')
    model_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_MODEL_FILE)
else:
    model_path = DRIVE_MODEL_PATH

if not os.path.exists(model_path):
    raise FileNotFoundError(f'Model not found: {model_path}\nCheck your model configuration in Step 2.')

ext = os.path.splitext(model_path)[1].lower()
if ext == '.tflite':
    MODEL_TYPE = 'tflite'
elif ext == '.onnx':
    MODEL_TYPE = 'onnx'
else:
    raise ValueError(f"Unsupported model format: '{ext}'. File must end in .tflite or .onnx")

MODEL_NAME = os.path.splitext(os.path.basename(model_path))[0]

if MODEL_TYPE == 'tflite':
    from ai_edge_litert.interpreter import Interpreter as TFLiteInterpreter
    _clf_model = TFLiteInterpreter(model_path=model_path)
    _clf_model.allocate_tensors()
    in_shape  = _clf_model.get_input_details()[0]['shape']
    out_shape = _clf_model.get_output_details()[0]['shape']
    print('Classificador TFLite carregado.')
elif MODEL_TYPE == 'onnx':
    import onnxruntime as ort
    _clf_model = ort.InferenceSession(model_path)
    in_shape  = _clf_model.get_inputs()[0].shape
    out_shape = _clf_model.get_outputs()[0].shape
    print('Classificador ONNX carregado.')
print(f'  Caminho         : {model_path}')
print(f'  Formato entrada : {in_shape}')
print(f'  Formato saída   : {out_shape}')
if MODEL_TYPE == 'onnx':
    print(f'  Provedores      : {_clf_model.get_providers()}')

# --- Carrega rótulos ---
_DELIMITERS = {'tab': '\t', 'comma (,)': ',', 'semicolon (;)': ';', 'underscore (_)': '_'}
_labels_sep = _DELIMITERS.get(LABELS_DELIMITER, "\t")

if LABELS_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    _lrepo = HF_LABELS_REPO.strip() or HF_REPO_ID
    print(f'\nBaixando rótulos do HuggingFace: {_lrepo} / {HF_LABELS_FILE} ...')
    labels_path = hf_hub_download(repo_id=_lrepo, filename=HF_LABELS_FILE)
else:
    labels_path = DRIVE_LABELS_PATH

if not os.path.exists(labels_path):
    raise FileNotFoundError(f'Labels file not found: {labels_path}')

labels = []
with open(labels_path, 'r', encoding='utf-8') as f:
    _lines = f.readlines()
if LABELS_HAS_HEADER and _lines:
    _lines = _lines[1:]
for _line in _lines:
    _line = _line.strip()
    if not _line:
        continue
    if _labels_sep in _line:
        _parts = _line.split(_labels_sep)
        if LABELS_COLUMN_INDEX < len(_parts):
            labels.append(_parts[LABELS_COLUMN_INDEX].strip())
        else:
            print(f'  AVISO: coluna {LABELS_COLUMN_INDEX} não encontrada em: {_line!r}')
    else:
        labels.append(_line)

print(f'\nRótulos carregados : {len(labels)} classes')
print(f'  Primeiros 5      : {labels[:5]}')
print(f'  Nome do modelo   : {MODEL_NAME}')

# --- Lê metadados do banco de dados ---
if not os.path.exists(EMBEDDINGS_DB_PATH):
    raise FileNotFoundError(f'Embeddings database not found: {EMBEDDINGS_DB_PATH}')

with sqlite3.connect(EMBEDDINGS_DB_PATH) as _con:
    _db_meta = dict(_con.execute('SELECT key, value FROM metadata').fetchall())

# Suporte ao formato auricularia (novo) e ao formato legado do notebook.
DB_WINDOW_S       = float(_db_meta.get("window_duration_s", _db_meta.get("window_size_s", 5.0)))
DB_EMBEDDING_SIZE = int(_db_meta.get("embedding_size", 1536))
DB_SAMPLE_RATE    = int(_db_meta.get("sample_rate", 32000))
DB_OVERLAP        = float(_db_meta.get("overlap", 0.0))
DB_MODEL_NAME     = _db_meta.get("model_id", _db_meta.get("model_name", "unknown"))

# --- Carrega metadados por ponto (stream_id, UTC offset, coordenadas) ---
_sites_meta = {}
try:
    with sqlite3.connect(EMBEDDINGS_DB_PATH) as _sc:
        for _row in _sc.execute('SELECT site_name, stream_id, utc_offset, lat, lon FROM sites').fetchall():
            _sites_meta[_row[0]] = {'stream_id': _row[1], 'utc_offset': float(_row[2]),
                                    'lat': _row[3], 'lon': _row[4]}
except Exception:
    pass

print('\nBanco de embeddings:')
print(f'  Backbone       : {DB_MODEL_NAME}')
print(f'  Taxa amostral  : {DB_SAMPLE_RATE} Hz')
print(f'  Janela         : {DB_WINDOW_S} s')
print(f'  Tamanho emb.   : {DB_EMBEDDING_SIZE}')
print(f'  Sobreposição   : {DB_OVERLAP}')
if _sites_meta:
    print(f'  Pontos         : {len(_sites_meta)}')
    for _sn, _sm in sorted(_sites_meta.items()):
        _utc = _sm['utc_offset']
        _sid = _sm['stream_id']
        print(f"    {_sn}: stream_id='{_sid}', UTC{_utc:+.1f}")

---
## Passo 4 — Verificar Análises Já Concluídas

Esta célula varre o banco de embeddings e a pasta de resultados para identificar quais gravações
**já foram classificadas**. Essas serão **puladas** no Passo 5, para que você possa reexecutar
sem repetir trabalho já feito.

In [ ]:
#@title 🔎 Verificar análises já concluídas { display-mode: "form" }

import os
import re
import sqlite3
from datetime import datetime
from collections import defaultdict

def _parse_stem_datetime(stem, prefix=""):
    m = re.search(r"(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})", stem)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf"{re.escape(prefix)}_(\d{{8}})_(\d{{6}})", stem)
    else:
        m = re.search(r"(\d{8})_(\d{6})", stem)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

from datetime import timedelta as _td

def _get_result_path(site, stem):
    _utc_dt  = _parse_stem_datetime(stem, FILENAME_PREFIX)
    _utc_off = _sites_meta.get(site, {}).get('utc_offset', 0.0)
    dt       = _utc_dt + _td(hours=_utc_off) if _utc_dt else None
    if dt is not None:
        result_fn = f"{site}_{dt.strftime('%Y%m%d_%H%M%S')}.{MODEL_NAME}.results.txt"
    else:
        result_fn = f'{site}_{stem}.{MODEL_NAME}.results.txt'
    return os.path.join(DRIVE_RESULTS_DIR, site, result_fn)

def _detect_db_format(con):
    emb_cols = {row[1] for row in con.execute("PRAGMA table_info(embeddings)").fetchall()}
    if "chunk_index" in emb_cols and "site_name" in emb_cols:
        return "new"
    return "notebook"

os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)

# Enumera pares únicos de ponto/gravação no banco.
_file_keys = []
with sqlite3.connect(EMBEDDINGS_DB_PATH) as _con:
    DB_FORMAT = _detect_db_format(_con)
    if DB_FORMAT == "new":
        for (site_name, recording_id) in _con.execute(
            'SELECT DISTINCT site_name, recording_id FROM embeddings ORDER BY site_name, recording_id'
        ).fetchall():
            _file_keys.append(f"{site_name}/{recording_id}")
    else:
        _seen = set()
        for (key,) in _con.execute('SELECT key FROM embeddings ORDER BY key'):
            prefix = '/'.join(key.split('/')[:2])
            if prefix not in _seen:
                _seen.add(prefix)
                _file_keys.append(prefix)

to_process   = []
already_done = []
for fk in _file_keys:
    site, stem = fk.split('/', 1)
    if os.path.exists(_get_result_path(site, stem)):
        already_done.append(fk)
    else:
        to_process.append(fk)

# Resumo por ponto
_by_site = defaultdict(lambda: {"total": 0, "done": 0})
for fk in _file_keys:
    site = fk.split('/')[0]
    _by_site[site]['total'] += 1
for fk in already_done:
    site = fk.split('/')[0]
    _by_site[site]['done'] += 1

print(f'Pasta de resultados  : {DRIVE_RESULTS_DIR}')
print(f'Nome do modelo       : {MODEL_NAME}')
print(f'Formato do banco     : {DB_FORMAT}')
print(f'Total de gravações   : {len(_file_keys)}')
print(f'Já analisadas        : {len(already_done)}')
print(f'Restam para executar : {len(to_process)}')
print()
for site_name, counts in sorted(_by_site.items()):
    print(f"  {site_name}: {counts['done']}/{counts['total']} concluídas")

if already_done:
    print()
    print('Já concluídas (serão puladas):')
    for fk in already_done[:8]:
        print(f'  {fk}')
    if len(already_done) > 8:
        print(f'  ... e mais {len(already_done) - 8}')

if not to_process:
    print()
    print('Todas as gravações já foram classificadas. Nada a fazer!')
    print('Delete os arquivos de resultado correspondentes para reclassificar.')
else:
    print()
    print(f'Pronto para classificar {len(to_process)} gravação(ões). Prossiga para o Passo 5.')

---
## Passo 5 — Executar Classificação e Salvar Resultados

Para cada gravação no banco de embeddings:
1. Todos os vetores de embedding pré-computados para aquela gravação são carregados
2. Cada embedding é passado pelo modelo classificador
3. Detecções acima do limiar de confiança são mantidas (até Top K por segmento)
4. Os resultados são salvos imediatamente no seu Google Drive

**Arquivos de resultado** são salvos em:
```
PASTA_RESULTADOS / NOME_PONTO / NOME_PONTO_AAAAMMDD_HHMMSS.NOME_MODELO.results.txt
```

Colunas: `site`, `lat`, `lon`, `date`, `time`, `start_time`, `end_time`, `label`, `confidence`

> Dependendo do número de gravações e segmentos, esta etapa pode demorar um pouco. O progresso é mostrado abaixo.

In [ ]:
#@title 🚀 Executar classificação { display-mode: "form" }

import csv
import os
import re
import sqlite3
import time
import numpy as np
from datetime import datetime, timedelta

def _parse_stem_datetime(stem, prefix=""):
    m = re.search(r"(\d{4})-(\d{2})-(\d{2})_(\d{2})-(\d{2})-(\d{2})", stem)
    if m:
        return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3)),
                        int(m.group(4)), int(m.group(5)), int(m.group(6)))
    if prefix:
        m = re.search(rf"{re.escape(prefix)}_(\d{{8}})_(\d{{6}})", stem)
    else:
        m = re.search(r"(\d{8})_(\d{6})", stem)
    if m:
        d, t = m.group(1), m.group(2)
        return datetime(int(d[:4]), int(d[4:6]), int(d[6:8]),
                        int(t[:2]), int(t[2:4]), int(t[4:6]))
    return None

def _apply_activation(logits, activation, sensitivity=-1.0, bias=1.0):
    arr = np.array(logits, dtype=np.float32).flatten()
    if activation == 'sigmoid':
        transformed_bias = (bias - 1.0) * 10.0
        return 1.0 / (1.0 + np.exp(sensitivity * np.clip(arr + transformed_bias, -20, 20)))
    elif activation == 'softmax':
        arr = arr - arr.max()
        e = np.exp(arr)
        return e / e.sum()
    return arr

def _run_classifier(emb):
    if MODEL_TYPE == 'tflite':
        in_det  = _clf_model.get_input_details()[0]
        out_det = _clf_model.get_output_details()[0]
        _clf_model.resize_tensor_input(in_det['index'], [1, len(emb)])
        _clf_model.allocate_tensors()
        _clf_model.set_tensor(in_det['index'], emb[np.newaxis, :])
        _clf_model.invoke()
        return _clf_model.get_tensor(out_det['index']).flatten()
    else:
        return _clf_model.run(
            [ONNX_OUTPUT_NAME],
            {ONNX_INPUT_NAME: emb[np.newaxis, :].astype(np.float32)},
        )[0].flatten()

def _get_result_path(site, stem):
    _utc_dt  = _parse_stem_datetime(stem, FILENAME_PREFIX)
    _utc_off = _sites_meta.get(site, {}).get('utc_offset', 0.0)
    dt       = _utc_dt + timedelta(hours=_utc_off) if _utc_dt else None
    if dt is not None:
        result_fn = f"{site}_{dt.strftime('%Y%m%d_%H%M%S')}.{MODEL_NAME}.results.txt"
    else:
        result_fn = f'{site}_{stem}.{MODEL_NAME}.results.txt'
    return os.path.join(DRIVE_RESULTS_DIR, site, result_fn)

if not to_process:
    print('Nenhuma gravação para classificar. Todas as análises já estão concluídas.')
else:
    print(f'Classificando {len(to_process)} gravação(ões) com o modelo "{MODEL_NAME}"')
    print(f'  Limiar de confiança : {MIN_CONFIDENCE}')
    print(f'  Top K               : {TOP_K}')
    print(f'  Tamanho da janela   : {DB_WINDOW_S} s')
    print(f'  Backbone            : {DB_MODEL_NAME}')
    print(f'  Formato do banco    : {DB_FORMAT}')
    print('=' * 65)

    total_detections = 0
    total_start      = time.time()

    # Pré-computa o passo para o formato auricularia (chunk_index → start_time).
    _stride_samples = max(1, int(DB_WINDOW_S * DB_SAMPLE_RATE * (1.0 - DB_OVERLAP)))

    with sqlite3.connect(EMBEDDINGS_DB_PATH) as _con:
        for rec_idx, fk in enumerate(to_process, 1):
            site, stem      = fk.split('/', 1)
            _utc_dt        = _parse_stem_datetime(stem, FILENAME_PREFIX)
            _sm            = _sites_meta.get(site, {})
            _utc_off       = _sm.get('utc_offset', 0.0)
            stream_id      = _sm.get('stream_id', '')
            utc_offset_str = f'UTC{_utc_off:+.1f}'
            file_dt        = _utc_dt + timedelta(hours=_utc_off) if _utc_dt else None
            site_lat       = _sm.get('lat', '') or str(_latlon_map.get(site, ('',''))[0])
            site_lon       = _sm.get('lon', '') or str(_latlon_map.get(site, ('',''))[1])

            print(f'[{rec_idx}/{len(to_process)}] {stem}  (ponto: {site})')

            if DB_FORMAT == "new":
                _emb_rows = _con.execute(
                    "SELECT chunk_index, data FROM embeddings "
                    "WHERE site_name = ? AND recording_id = ? ORDER BY chunk_index",
                    (site, stem)
                ).fetchall()
            else:
                _emb_rows = _con.execute(
                    "SELECT key, data FROM embeddings WHERE key LIKE ? ORDER BY key",
                    (fk + "/%",)
                ).fetchall()

            rows       = []
            file_start = time.time()

            for row_key, data in _emb_rows:
                if DB_FORMAT == "new":
                    start_s = int(row_key) * _stride_samples / DB_SAMPLE_RATE
                else:
                    start_s = float(row_key.split('/')[2])
                end_s = start_s + DB_WINDOW_S
                emb   = np.frombuffer(data, dtype=np.float32).copy()

                try:
                    logits = _run_classifier(emb)
                except Exception as e:
                    print(f'  ERRO em {start_s:.1f}s: {e}')
                    continue

                scores = _apply_activation(logits, ACTIVATION, SIGMOID_SENSITIVITY, SIGMOID_BIAS)

                _seg_hits = sorted(
                    [(float(scores[i]), i)
                     for i in range(min(len(scores), len(labels)))
                     if float(scores[i]) >= MIN_CONFIDENCE],
                    reverse=True
                )[:TOP_K]

                for score, idx in _seg_hits:
                    date_str = file_dt.strftime("%Y-%m-%d") if file_dt else ""
                    time_str = file_dt.strftime("%H:%M:%S") if file_dt else ""
                    rows.append([
                        stream_id, site, site_lat, site_lon,
                        date_str, time_str, utc_offset_str,
                        f"{start_s:.1f}", f"{end_s:.1f}",
                        labels[idx], f"{float(score):.4f}"
                    ])

            result_path = _get_result_path(site, stem)
            os.makedirs(os.path.dirname(result_path), exist_ok=True)
            with open(result_path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['stream_id', 'site', 'lat', 'lon', 'date', 'time', 'utc_offset', 'start_time', 'end_time', 'label', 'confidence'])
                writer.writerows(rows)

            elapsed  = time.time() - file_start
            n_segs   = len(_emb_rows)
            per_ms   = elapsed / n_segs * 1000 if n_segs else 0
            total_detections += len(rows)
            print(f'  → {len(rows)} detecção(ões)  |  {n_segs} segmentos  |  '
                  f'{elapsed:.1f}s ({per_ms:.0f}ms/seg)  →  {os.path.basename(result_path)}')

    total_elapsed = time.time() - total_start
    print()
    print('=' * 65)
    print('Classificação concluída!')
    print(f'  Gravações classificadas : {len(to_process)}')
    print(f'  Total de detecções      : {total_detections}')
    print(f'  Tempo total             : {total_elapsed:.1f}s')
    print(f'  Resultados salvos em    : {DRIVE_RESULTS_DIR}')

---
## Passo 6 — Mesclar Resultados em um Único CSV

O Passo 5 salva um arquivo de resultado por gravação (para que uma execução interrompida possa retomar).
Este passo combina todos esses arquivos em um **único CSV** no seu Google Drive:
```
PASTA_RESULTADOS / MODEL_NAME.merged.results.csv
```
Colunas: `stream_id`, `site`, `lat`, `lon`, `date`, `time`, `utc_offset`, `start_time`, `end_time`, `label`, `confidence`

Carregue este arquivo diretamente no notebook **Análise de Resultados de Classificação** apontando a
configuração *Pasta de resultados* para este arquivo `.merged.results.csv`.

> Os arquivos individuais por gravação **não** são apagados — você pode reexecutar este passo quando quiser.

In [ ]:
#@title 🧩 Mesclar todos os resultados em um único CSV { display-mode: "form" }

#@markdown Combina todos os arquivos de resultado por gravação da pasta de resultados em um único CSV,
#@markdown pronto para carregar no notebook **Análise de Resultados de Classificação**.
#@markdown Os arquivos individuais por gravação são mantidos para que uma execução interrompida possa retomar.

import os
import csv
import glob

MERGED_CSV_PATH = os.path.join(DRIVE_RESULTS_DIR, f'{MODEL_NAME}.merged.results.csv')

# Encontra todos os arquivos de resultado por gravação (busca recursiva, um por subpasta de ponto).
_result_files = sorted(glob.glob(
    os.path.join(DRIVE_RESULTS_DIR, '**', f'*.{MODEL_NAME}.results.txt'), recursive=True))

if not _result_files:
    print(f'Nenhum arquivo de resultado encontrado em: {DRIVE_RESULTS_DIR}')
    print('Execute o passo de análise primeiro para gerar os resultados por gravação.')
else:
    # Processa arquivo por arquivo para nunca manter todos os resultados na memória de uma vez.
    _header  = None
    _n_files = 0
    _n_rows  = 0
    with open(MERGED_CSV_PATH, 'w', newline='', encoding='utf-8') as _out:
        _writer = csv.writer(_out)
        for _fp in _result_files:
            if os.path.abspath(_fp) == os.path.abspath(MERGED_CSV_PATH):
                continue  # nunca mesclar o arquivo de saída nele mesmo
            try:
                with open(_fp, 'r', newline='', encoding='utf-8') as _in:
                    _rows = list(csv.reader(_in))
            except Exception as _e:
                print(f'  AVISO: não foi possível ler {os.path.basename(_fp)}: {_e}')
                continue
            if not _rows:
                continue
            _file_header, _data = _rows[0], _rows[1:]
            if _header is None:
                _header = _file_header
                _writer.writerow(_header)
            _writer.writerows(_data)
            _n_files += 1
            _n_rows  += len(_data)

    print('Mesclagem concluída!')
    print(f'  Arquivos mesclados   : {_n_files}')
    print(f'  Total de detecções   : {_n_rows}')
    print(f'  CSV salvo em         : {MERGED_CSV_PATH}')